# CLM losses — sensibilidad del panel T-gen en F3 (post hoc)

**rev. A1 · 2026-10-07** · código clm_losses 0.3.7 (hash `848167f1c158ebed`, el mismo del notebook principal rev. 4.7) + script `analisis/F3_sensibilidad/sensibilidad_panel.py` (sha256 `c5a2a566c9b6`).

**Qué hace.** Lee las corridas de F3 que ya están en tu Drive y repite la parte T-gen de C1 y C2 cambiando qué generadores entran al macro:

- los 5 congelados: es el control, tiene que dar lo mismo que el confirmatorio;
- sin cada uno de ellos;
- cada uno solo.

**Qué no hace.** No entrena, no evalúa, no descarga datos, no necesita GPU y no modifica ninguna corrida. Sólo escribe `results/F3/sensibilidad_panel/`.

**Por qué.** En el test confirmatorio, la auditoría de Phi-4 dio 0.312 (umbral 0.30). Además, la ventaja de InfoNCE sobre VICReg es muy distinta según el generador (EXP-011). El macro congelado se mantiene; este análisis es descriptivo y post hoc, y así se reporta.

**Cómo.** Entorno de ejecución → **Ejecutar todo** (basta con CPU). Tarda ~5 min. Al terminar, sube el notebook ejecutado.

In [ ]:
#@title A.1 · Parámetros y Drive (fijos: no hay nada que cambiar)
import os, sys
PROFILE = "colab"
DRIVE_BASE = "/content/drive/MyDrive/CLM_losses"
CACHE_ROOT = "/content/cache"
USE_DRIVE_CODE = False
from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(os.path.join(DRIVE_BASE, PROFILE)):
    raise SystemExit(f"No encuentro {DRIVE_BASE}/{PROFILE} en tu Drive: abre este notebook con la cuenta del estudio.")
print("Carpeta del estudio:", os.path.join(DRIVE_BASE, PROFILE))

In [ ]:
#@title A.2 · Instalar el código del estudio (el mismo del notebook principal)
import base64, hashlib, io, shutil, zipfile
CODE_ZIP_B64 = "..."
CODE_ZIP_SHA256 = "cb7250acbb3df3dd0920d7f5974f843a2c89d2bf43c8a06906eb3997bbd04fbe"
PKG_ROOT = "/content/clm_losses_pkg"
drive_zip = os.path.join(DRIVE_BASE, "code", "clm_losses_pkg.zip")
if USE_DRIVE_CODE and os.path.exists(drive_zip):
    data, origen = open(drive_zip, "rb").read(), drive_zip
else:
    data, origen = base64.b64decode(CODE_ZIP_B64), "embebido"
    assert hashlib.sha256(data).hexdigest() == CODE_ZIP_SHA256, "el código embebido está corrupto"
shutil.rmtree(PKG_ROOT, ignore_errors=True)
zipfile.ZipFile(io.BytesIO(data)).extractall(PKG_ROOT)
open("/content/clm_code.zip", "wb").write(data)       # para rehacer la Sección 1 sola si la sesión se reinicia
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)
os.environ["PYTHONPATH"] = PKG_ROOT + os.pathsep + os.environ.get("PYTHONPATH", "")
for m in [m for m in list(sys.modules) if m == "clm_losses" or m.startswith("clm_losses.")]:
    del sys.modules[m]
import clm_losses
from clm_losses.utils import code_hash
CODE_HASH = code_hash()
snap = os.path.join(DRIVE_BASE, "code", CODE_HASH)
if not os.path.exists(snap):
    shutil.copytree(os.path.join(PKG_ROOT, "clm_losses"), snap)
print(f"clm_losses {clm_losses.__version__} ({origen}) · hash {CODE_HASH} · copia legible: {snap}")

In [ ]:
#@title A.3 · Sensibilidad del panel T-gen en F3 (post hoc, descriptivo; sólo lee las corridas)
import base64, hashlib, importlib.util
import pandas as pd
from IPython.display import Markdown, display
from clm_losses.pipeline import freeze_path, setup
SCRIPT_B64 = "IiIiU2Vuc2liaWxpZGFkIGRlbCBwYW5lbCBULWdlbiBlbiBGMyAocG9zdCBob2MsIGRlc2NyaXB0aXZvOyBFWFAtMDExIHkgRFAtMDggZGVsIEVYUEVSSU1FTlRfTE9HKS4KClBvciBxdcOpOiBlbiBlbCB0ZXN0IGNvbmZpcm1hdG9yaW8gbGEgYXVkaXRvcsOtYSBkZSBQaGktNCBkaW8gMC4zMTIgKHVtYnJhbCAwLjMwOiBiYW5kYSBhbHRhKSB5IGxhIGRpZmVyZW5jaWEKVklDUmVnIOKIkiBJbmZvTkNFIHBvciBnZW5lcmFkb3IgZXMgaGV0ZXJvZ8OpbmVhLiBMYSByZWdsYSBkZWwgZnJlZXplIG1hbmRhIHVzYXIgZWwgbWFjcm8gY29uZ2VsYWRvIGF1bnF1ZSBzdQphdWRpdG9yw61hIGNhbWJpZSwgeSBlc28gbm8gc2UgdG9jYTogQzEgeSBDMiBjb25maXJtYXRvcmlvcyBxdWVkYW4gY29tbyBlc3TDoW4uIEFxdcOtIHPDs2xvIHNlIGRlc2NyaWJlIGN1w6FudG8KZGVwZW5kZW4gZGUgY2FkYSBnZW5lcmFkb3I6CgotIMKrbG9zIE7CuzogZWwgbWFjcm8gY29uZ2VsYWRvLiBFcyBlbCBjb250cm9sOiB0aWVuZSBxdWUgZGFyIGV4YWN0YW1lbnRlIGxvIG1pc21vIHF1ZSBlbCBjb25maXJtYXRvcmlvOwotIMKrc2luIFjCuzogZWwgbWFjcm8gZGUgbG9zIG90cm9zIGdlbmVyYWRvcmVzLCBlbiBsYSBpbnRlcnNlY2Npw7NuIGRlIHN1cyDDrXRlbXM7Ci0gwqtzw7NsbyBYwrs6IHVuIGdlbmVyYWRvciBjb24gdG9kb3Mgc3VzIMOtdGVtcy4KClVzYSBlbCBtaXNtbyBib290c3RyYXAgcGFyZWFkbyBwb3IgZG9jdW1lbnRvIHF1ZSBlbCBjb25maXJtYXRvcmlvIChzdGF0cy5jb250cmFzdCwgc2VtaWxsYSAwKS4gTm8gbGxldmEgSG9sbToKbm8gZXMgdW5hIHBydWViYS4gU8OzbG8gbGVlIGxhcyBjb3JyaWRhcyB5IGVzY3JpYmUgcmVzdWx0cy88ZmFzZT4vc2Vuc2liaWxpZGFkX3BhbmVsL3tzZW5zaWJpbGlkYWQuanNvbiwKU0VOU0lCSUxJREFELm1kfS4gUmVxdWllcmUgZWwgcGFxdWV0ZSBjbG1fbG9zc2VzIHYwLjMuNiAobm8gbG8gbW9kaWZpY2EpLgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG9zCgppbXBvcnQgbnVtcHkgYXMgbnAKCmZyb20gY2xtX2xvc3Nlcy5hbmFseXNpcyBpbXBvcnQgQzJfUEFJUlMsIF9hdCwgX3RnZW1fYXQKZnJvbSBjbG1fbG9zc2VzLnN0YXRzIGltcG9ydCBjbHVzdGVyc19mcm9tX2RvY3MsIGNvbnRyYXN0CgpNT1RJVk8gPSAoIlBvc3QgaG9jIHkgZGVzY3JpcHRpdm8uIEVuIGVsIHRlc3QgY29uZmlybWF0b3JpbyBsYSBhdWRpdG9yw61hIGRlIFBoaS00IGRpbyAwLjMxMiAodW1icmFsIDAuMzA6IGJhbmRhICIKICAgICAgICAgICJhbHRhKSB5IGxhIGRpZmVyZW5jaWEgVklDUmVnIOKIkiBJbmZvTkNFIHBvciBnZW5lcmFkb3IgZXMgaGV0ZXJvZ8OpbmVhLiBFbCBtYWNybyBjb25nZWxhZG8gc2UgbWFudGllbmUgIgogICAgICAgICAgIihyZWdsYSBkZWwgZnJlZXplKTsgZXN0byBzw7NsbyBkZXNjcmliZSBjdcOhbnRvIGRlcGVuZGVuIEMxIChwYXJ0ZSBULWdlbikgeSBDMiBkZSBjYWRhIGdlbmVyYWRvci4iKQoKCmRlZiBzdWJzZXRzKGtleXMpIC0+IGxpc3Q6CiAgICAiIiJbKG5vbWJyZSwgZ2VuZXJhZG9yZXMpXTogZWwgbWFjcm8gY29tcGxldG8sIGNhZGEgdW5vIGFmdWVyYSB5IGNhZGEgdW5vIHNvbG8uIiIiCiAgICBrZXlzID0gbGlzdChrZXlzKQogICAgb3V0ID0gWyhmImxvcyB7bGVuKGtleXMpfSAobWFjcm8gY29uZ2VsYWRvKSIsIGtleXMpXQogICAgaWYgbGVuKGtleXMpID4gMToKICAgICAgICBvdXQgKz0gWyhmInNpbiB7a30iLCBbeCBmb3IgeCBpbiBrZXlzIGlmIHggIT0ga10pIGZvciBrIGluIGtleXNdCiAgICAgICAgb3V0ICs9IFsoZiJzw7NsbyB7a30iLCBba10pIGZvciBrIGluIGtleXNdCiAgICByZXR1cm4gb3V0CgoKZGVmIGxlY3R1cmFfYzEoYzogZGljdCwgcm9wZTogZmxvYXQsIGE6IHN0ciwgYjogc3RyID0gImluZm9uY2UiKSAtPiBzdHI6CiAgICAiIiJNaXNtYSByZWdsYSBxdWUgZWwgdmVyZWRpY3RvIGRlIEMxLCBtw6FzIHNpIGVsIElDOTUgcXVlZGEgZW50ZXJvIGZ1ZXJhIGRlIMKxUk9QRS4iIiIKICAgIGlmIGNbInAiXSA8IDAuMDUgYW5kIGFicyhjWyJkaWZmIl0pID49IHJvcGU6CiAgICAgICAgbWVqb3IgPSBmIntifSBtZWpvciIgaWYgY1siZGlmZiJdIDwgMCBlbHNlIGYie2F9IG1lam9yIgogICAgICAgIGZ1ZXJhID0gY1siY2k5NV9oaSJdIDwgLXJvcGUgaWYgY1siZGlmZiJdIDwgMCBlbHNlIGNbImNpOTVfbG8iXSA+IHJvcGUKICAgICAgICByZXR1cm4gbWVqb3IgKyAoIiAoSUM5NSBmdWVyYSBkZSDCsVJPUEUpIiBpZiBmdWVyYSBlbHNlICIgKElDOTUgZW50cmEgZW4gwrFST1BFKSIpCiAgICBpZiBjWyJwIl0gPCAwLjA1OgogICAgICAgIHJldHVybiAic2lnbmlmaWNhdGl2YSBwZXJvIG1lbm9yIHF1ZSBST1BFIgogICAgaWYgY1sidG9zdF9lcXVpdmFsZW50Il06CiAgICAgICAgcmV0dXJuICJlcXVpdmFsZW50ZXMgKFRPU1QgwrFST1BFKSIKICAgIHJldHVybiAibm8gY29uY2x1eWVudGUiCgoKZGVmIGNvbXB1dGUoZ3JvdXBzOiBkaWN0LCBrZXlzLCBpdGVtc19vZiwgZG9jX2lkcywgbmMsIHJvcGU6IGZsb2F0LCBhdCwgdGdlbV9hdCwgQjogaW50ID0gNDAwMCwKICAgICAgICAgICAgcGFpcnM9QzJfUEFJUlMpIC0+IGxpc3Q6CiAgICAiIiJOw7pjbGVvIHNpbiBsZWN0dXJhIG5pIGVzY3JpdHVyYSBkZSBhcmNoaXZvcy4KCiAgICBncm91cHM6IHticmF6bzogW2NhcnBldGFzIGRlIGNvcnJpZGFzXX07IGl0ZW1zX29mKGdlbmVyYWRvcmVzKSAtPiDDrXRlbXMgKMOtbmRpY2VzIGRlbCB0ZXN0KTsKICAgIGF0KGNhcnBldGEsIGdlbmVyYWRvciwgw610ZW1zKSAtPiBhY2llcnRvIHBvciDDrXRlbSBlbiBlbCBzZXQgZGUgZXNlIGdlbmVyYWRvciAoTm9uZSBzaSBmYWx0YSk7CiAgICB0Z2VtX2F0KGNhcnBldGEsIMOtdGVtcykgLT4gYWNpZXJ0byBwb3Igw610ZW0gZW4gVC1nZW0uIEVsIGFjaWVydG8gZGUgdW4gc3ViY29uanVudG8gZXMgZWwgcHJvbWVkaW8gcG9yIMOtdGVtCiAgICBzb2JyZSBzdXMgZ2VuZXJhZG9yZXMsIGNvbW8gZWwgbWFjcm8gZGUgVC1nZW4uIiIiCiAgICBkb2NfaWRzID0gbnAuYXNhcnJheShkb2NfaWRzKQogICAgcm93cyA9IFtdCiAgICBmb3IgbmFtZSwga3MgaW4gc3Vic2V0cyhrZXlzKToKICAgICAgICBpdGVtcyA9IG5wLmFzYXJyYXkoaXRlbXNfb2Yoa3MpLCBucC5pbnQ2NCkKICAgICAgICByZWMgPSB7InN1YmNvbmp1bnRvIjogbmFtZSwgImdlbmVyYWRvcmVzIjogbGlzdChrcyksICJuX2l0ZW1zIjogaW50KGxlbihpdGVtcykpfQogICAgICAgIGlmIGxlbihpdGVtcykgPT0gMDoKICAgICAgICAgICAgcm93cy5hcHBlbmQocmVjKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNsID0gY2x1c3RlcnNfZnJvbV9kb2NzKGRvY19pZHNbaXRlbXNdKQoKICAgICAgICBkZWYgbWF0KGFybSk6CiAgICAgICAgICAgIG91dCA9IFtdCiAgICAgICAgICAgIGZvciByZCBpbiBncm91cHMuZ2V0KGFybSwgW10pOgogICAgICAgICAgICAgICAgY29scyA9IFthdChyZCwgaywgaXRlbXMpIGZvciBrIGluIGtzXQogICAgICAgICAgICAgICAgaWYgYW55KGMgaXMgTm9uZSBmb3IgYyBpbiBjb2xzKToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICAgICAgb3V0LmFwcGVuZChucC5tZWFuKGNvbHMsIDApLmFzdHlwZShucC5mbG9hdDMyKSkKICAgICAgICAgICAgcmV0dXJuIG5wLnZzdGFjayhvdXQpIGlmIG91dCBlbHNlIE5vbmUKCiAgICAgICAgZGVmIHRtYXQoYXJtKToKICAgICAgICAgICAgb3V0ID0gW3RnZW1fYXQocmQsIGl0ZW1zKSBmb3IgcmQgaW4gZ3JvdXBzLmdldChhcm0sIFtdKV0KICAgICAgICAgICAgcmV0dXJuIG5wLnZzdGFjayhvdXQpLmFzdHlwZShucC5mbG9hdDMyKSBpZiBvdXQgYW5kIGFsbChvIGlzIG5vdCBOb25lIGZvciBvIGluIG91dCkgZWxzZSBOb25lCgogICAgICAgIE0gPSB7YXJtOiBtYXQoYXJtKSBmb3IgYXJtIGluIGdyb3Vwc30KICAgICAgICBpZiBuYyBhbmQgTS5nZXQobmMpIGlzIG5vdCBOb25lIGFuZCBNLmdldCgiaW5mb25jZSIpIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjID0gY29udHJhc3QoTVtuY10sIE1bImluZm9uY2UiXSwgY2wsIHJvcGUsIEI9QikKICAgICAgICAgICAgYy51cGRhdGUoQT1uYywgQj0iaW5mb25jZSIsIGxlY3R1cmE9bGVjdHVyYV9jMShjLCByb3BlLCBuYykpCiAgICAgICAgICAgIHJlY1siQzEiXSA9IGMKICAgICAgICBmb3IgYSwgYiBpbiBwYWlyczoKICAgICAgICAgICAgVGEsIFRiID0gdG1hdChhKSwgdG1hdChiKQogICAgICAgICAgICBpZiBhbnkoeCBpcyBOb25lIGZvciB4IGluIChNLmdldChhKSwgTS5nZXQoYiksIFRhLCBUYikpOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgcmVjW2Yie2F9IOKIkiB7Yn0iXSA9IHsizpQgVC1nZW0iOiBjb250cmFzdChUYSwgVGIsIGNsLCByb3BlLCBCPUIpLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAizpQgVC1nZW4iOiBjb250cmFzdChNW2FdLCBNW2JdLCBjbCwgcm9wZSwgQj1CKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIkMyIjogY29udHJhc3QoVGEgLSBNW2FdLCBUYiAtIE1bYl0sIGNsLCByb3BlLCBCPUIpfQogICAgICAgIHJvd3MuYXBwZW5kKHJlYykKICAgIHJldHVybiByb3dzCgoKZGVmIF9wcChjOiBkaWN0IHwgTm9uZSkgLT4gc3RyOgogICAgaWYgbm90IGM6CiAgICAgICAgcmV0dXJuICLigJQiCiAgICByZXR1cm4gZiJ7MTAwICogY1snZGlmZiddOisuMmZ9IFt7MTAwICogY1snY2k5NV9sbyddOisuMmZ9LCB7MTAwICogY1snY2k5NV9oaSddOisuMmZ9XSIKCgpkZWYgY29udHJvbChyb3dzOiBsaXN0LCBjMV9jb25mOiBkaWN0IHwgTm9uZSwgYzJfY29uZjogbGlzdCB8IE5vbmUsIHRvbDogZmxvYXQgPSAxZS05KSAtPiBkaWN0OgogICAgIiIiwr/Cq2xvcyBOwrsgcmVwcm9kdWNlIGVsIGNvbmZpcm1hdG9yaW8/IFBhcmEgQzEgKFQtZ2VuKSB5IEMyIHBvciBwYXI6IGxhIGRpZmVyZW5jaWEgcHVudHVhbCB0aWVuZSBxdWUgc2VyCiAgICBpZMOpbnRpY2E7IGVsIElDOTUgdGFtYmnDqW4sIHNpIGVsIGNvbmZpcm1hdG9yaW8gdXPDsyBlbCBtaXNtbyBuw7ptZXJvIGRlIHJlbXVlc3RyZW9zIChCPTQwMDAgZW4gZWwgZXN0dWRpbyByZWFsOwogICAgZWwgc21va2UgdXNhIG1lbm9zLCB5IGFow60gc8OzbG8gY29pbmNpZGUgZWwgcHVudG8pLiIiIgogICAgZmlyc3QgPSByb3dzWzBdIGlmIHJvd3MgZWxzZSB7fQogICAgb3V0ID0ge30KCiAgICBkZWYgY21wKG1pbmUsIGNvbmYpOgogICAgICAgIHJldHVybiB7InB1bnRvIjogYWJzKG1pbmVbImRpZmYiXSAtIGNvbmZbImRpZmYiXSkgPD0gdG9sLAogICAgICAgICAgICAgICAgIklDOTUiOiBhbGwoYWJzKG1pbmVba10gLSBjb25mW2tdKSA8PSB0b2wgZm9yIGsgaW4gKCJjaTk1X2xvIiwgImNpOTVfaGkiKSl9CiAgICBpZiBjMV9jb25mIGFuZCBmaXJzdC5nZXQoIkMxIik6CiAgICAgICAgb3V0WyJDMSJdID0gY21wKGZpcnN0WyJDMSJdLCBjMV9jb25mKQogICAgZm9yIHIgaW4gYzJfY29uZiBvciBbXToKICAgICAgICBtaW5lID0gKGZpcnN0LmdldChyWyJwYXIiXSkgb3Ige30pLmdldCgiQzIiKQogICAgICAgIGlmIG1pbmU6CiAgICAgICAgICAgIG91dFtmIkMyIHtyWydwYXInXX0iXSA9IGNtcChtaW5lLCByWyJDMiJdKQogICAgcmV0dXJuIG91dAoKCmRlZiBfY29udHJvbF90eHQoY3RybDogZGljdCB8IE5vbmUpIC0+IHN0cjoKICAgIGlmIG5vdCBjdHJsOgogICAgICAgIHJldHVybiAibm8gaGF5IHJlc3VsdGFkb3MgY29uZmlybWF0b3Jpb3MgY29uIHF1w6kgY29tcGFyYXIiCiAgICBzaSA9IGxhbWJkYSB4OiAic8OtIiBpZiB4IGVsc2UgIk5PIgogICAgcmV0dXJuICI7ICIuam9pbihmIntrfTogcHVudG8ge3NpKHZbJ3B1bnRvJ10pfSwgSUM5NSB7c2kodlsnSUM5NSddKX0iIGZvciBrLCB2IGluIGN0cmwuaXRlbXMoKSkKCgpkZWYgdG9fbWFya2Rvd24ocmVzOiBkaWN0KSAtPiBzdHI6CiAgICByb3dzID0gcmVzLmdldCgiZmlsYXMiLCBbXSkKICAgIHBhcmVzID0gW2Yie2F9IOKIkiB7Yn0iIGZvciBhLCBiIGluIEMyX1BBSVJTXQogICAgbmMgPSByZXMuZ2V0KCJicmF6b19DMSIpCiAgICBsaW5lcyA9IFtmIiMgU2Vuc2liaWxpZGFkIGRlbCBwYW5lbCBULWdlbiDigJQge3Jlcy5nZXQoJ2Zhc2UnKX0gKHtyZXMuZ2V0KCdzcGxpdCcpfSkg4oCUIHtyZXMuZ2V0KCdjcmVhZG8nKX0iLCAiIiwKICAgICAgICAgICAgIHJlcy5nZXQoIm1vdGl2byIsICIiKSwgIiIsCiAgICAgICAgICAgICBmIlBhbmVsIHtyZXMuZ2V0KCdwYW5lbF92ZXJzaW9uJyl9IMK3IG1hY3JvIHtyZXMuZ2V0KCdtYWNybycpfSDCtyBjb3JyaWRhcyBwb3IgYnJhem8ge3Jlcy5nZXQoJ2NvcnJpZGFzJyl9IiwKICAgICAgICAgICAgICIiLCAiQ29udHJvbCAobGEgZmlsYSBkZWwgbWFjcm8gY29uZ2VsYWRvIHJlcHJvZHVjZSBlbCBjb25maXJtYXRvcmlvKTogIiArIF9jb250cm9sX3R4dChyZXMuZ2V0KCJjb250cm9sIikpLCAiIiwKICAgICAgICAgICAgICJEaWZlcmVuY2lhcyBlbiBwcCwgSUM5NSBwb3IgYm9vdHN0cmFwIHBhcmVhZG8gcG9yIGRvY3VtZW50byAoc2luIEhvbG06IGVzIGRlc2NyaXB0aXZvKS4iLCAiIiwKICAgICAgICAgICAgIGYifCBTdWJjb25qdW50byB8IMOtdGVtcyB8IEMxOiB7bmN9IOKIkiBpbmZvbmNlIHwgbGVjdHVyYSB8ICIKICAgICAgICAgICAgICsgIiB8ICIuam9pbihmIkMyIHtwfSAozpQgVC1nZW4pIiBmb3IgcCBpbiBwYXJlcykgKyAiIHwiLAogICAgICAgICAgICAgInwtLS18LS0tfC0tLXwtLS18IiArICItLS18IiAqIGxlbihwYXJlcyldCiAgICBmb3IgciBpbiByb3dzOgogICAgICAgIGMycyA9IFtdCiAgICAgICAgZm9yIHAgaW4gcGFyZXM6CiAgICAgICAgICAgIGQgPSByLmdldChwKQogICAgICAgICAgICBjMnMuYXBwZW5kKGYie19wcChkWydDMiddKX0gKHsxMDAgKiBkWyfOlCBULWdlbiddWydkaWZmJ106Ky4xZn0pIiBpZiBkIGVsc2UgIuKAlCIpCiAgICAgICAgbGluZXMuYXBwZW5kKGYifCB7clsnc3ViY29uanVudG8nXX0gfCB7clsnbl9pdGVtcyddfSB8IHtfcHAoci5nZXQoJ0MxJykpfSB8ICIKICAgICAgICAgICAgICAgICAgICAgZiJ7KHIuZ2V0KCdDMScpIG9yIHt9KS5nZXQoJ2xlY3R1cmEnLCAn4oCUJyl9IHwgIiArICIgfCAiLmpvaW4oYzJzKSArICIgfCIpCiAgICByZXR1cm4gIlxuIi5qb2luKGxpbmVzKSArICJcbiIKCgpkZWYgdGFibGEocmVzOiBkaWN0KToKICAgICIiIkxhIG1pc21hIHRhYmxhIGNvbW8gRGF0YUZyYW1lIChwcCkuIiIiCiAgICBpbXBvcnQgcGFuZGFzIGFzIHBkCiAgICBvdXQgPSBbXQogICAgZm9yIHIgaW4gcmVzLmdldCgiZmlsYXMiLCBbXSk6CiAgICAgICAgcm93ID0geyJzdWJjb25qdW50byI6IHJbInN1YmNvbmp1bnRvIl0sICLDrXRlbXMiOiByWyJuX2l0ZW1zIl19CiAgICAgICAgYyA9IHIuZ2V0KCJDMSIpCiAgICAgICAgaWYgYzoKICAgICAgICAgICAgcm93LnVwZGF0ZSh7IkMxIM6UIjogcm91bmQoMTAwICogY1siZGlmZiJdLCAyKSwgIkMxIElDOTUiOiBmIlt7MTAwICogY1snY2k5NV9sbyddOi4yZn0sIHsxMDAgKiBjWydjaTk1X2hpJ106LjJmfV0iLAogICAgICAgICAgICAgICAgICAgICAgICAiQzEgbGVjdHVyYSI6IGNbImxlY3R1cmEiXX0pCiAgICAgICAgZm9yIGEsIGIgaW4gQzJfUEFJUlM6CiAgICAgICAgICAgIGQgPSByLmdldChmInthfSDiiJIge2J9IikKICAgICAgICAgICAgaWYgZDoKICAgICAgICAgICAgICAgIHJvd1tmIkMyIHthfSJdID0gcm91bmQoMTAwICogZFsiQzIiXVsiZGlmZiJdLCAyKQogICAgICAgICAgICAgICAgcm93W2YizpQgVC1nZW4ge2F9Il0gPSByb3VuZCgxMDAgKiBkWyLOlCBULWdlbiJdWyJkaWZmIl0sIDIpCiAgICAgICAgb3V0LmFwcGVuZChyb3cpCiAgICByZXR1cm4gcGQuRGF0YUZyYW1lKG91dCkKCgpkZWYgcGFuZWxfc2Vuc2l0aXZpdHkoc3R1ZHksIHBoYXNlOiBzdHIgPSAiRjMiLCBzcGxpdDogc3RyID0gImNvbmZpcm0iLCBCOiBpbnQgPSA0MDAwLCBsb2c9cHJpbnQsCiAgICAgICAgICAgICAgICAgICAgICBtZXRhOiBkaWN0IHwgTm9uZSA9IE5vbmUpIC0+IGRpY3Q6CiAgICAiIiJMZWUgZWwgcGFuZWwgY29uZ2VsYWRvIGRlbCBzcGxpdCwgbGFzIGNvcnJpZGFzIGRlIGxhIGZhc2UgeSBsb3Mgw610ZW1zOyBjYWxjdWxhIHkgZ3VhcmRhIGxhIHNlbnNpYmlsaWRhZC4KICAgIG1ldGEgKHAuIGVqLiBlbCBoYXNoIGRlbCBjw7NkaWdvIHkgZGVsIHNjcmlwdCkgc2UgZ3VhcmRhIGp1bnRvIGNvbiBlbCByZXN1bHRhZG8uIiIiCiAgICBmcm9tIGNsbV9sb3NzZXMuZGF0YSBpbXBvcnQgRXZhbFNldAogICAgZnJvbSBjbG1fbG9zc2VzLmdlbmVyYXRvcnMgaW1wb3J0IF9pbnRlcnNlY3QsIGxvYWRfcGFuZWwKICAgIGZyb20gY2xtX2xvc3Nlcy5waXBlbGluZSBpbXBvcnQgZnJlZXplX3BhdGgsIGl0ZXJfcnVuX2RpcnMKICAgIGZyb20gY2xtX2xvc3Nlcy51dGlscyBpbXBvcnQgbG9hZF9qc29uLCBtYWtlZGlycywgbm93X2lzbywgc2F2ZV9qc29uCiAgICBwYW5lbCA9IGxvYWRfcGFuZWwoc3R1ZHksIHNwbGl0KSBvciB7fQogICAga2V5cyA9IGxpc3QocGFuZWwuZ2V0KCJtYWNybyIpIG9yIFtdKQogICAgaWYgbm90IGtleXM6CiAgICAgICAgbG9nKGYiW3NlbnNpYmlsaWRhZF0gbm8gaGF5IG1hY3JvIGRlIFQtZ2VuIGVuIHtzcGxpdH06IG5vIHNlIGhhY2UgbmFkYSIpCiAgICAgICAgcmV0dXJuIHt9CiAgICBuYyA9IChsb2FkX2pzb24oZnJlZXplX3BhdGgoc3R1ZHkpKSBvciB7fSkuZ2V0KCJDMV9iZXN0X25vbmNvbnRyYXN0aXZlIikKICAgIGFybXMgPSBzb3J0ZWQoe25jLCAiaW5mb25jZSIsICpbeCBmb3IgcCBpbiBDMl9QQUlSUyBmb3IgeCBpbiBwXX0gLSB7Tm9uZX0pCiAgICBydW5zID0gaXRlcl9ydW5fZGlycyhzdHVkeSwgcGhhc2UpCiAgICBncm91cHMgPSB7YTogW3IgZm9yIHIgaW4gcnVucyBpZiBmIi97cGhhc2V9L3thfS8iIGluIHIucmVwbGFjZShvcy5zZXAsICIvIildIGZvciBhIGluIGFybXN9CiAgICBkb2NfaWRzID0gRXZhbFNldChzdHVkeS5wYXRocy5kKCJkYXRhIiwgImV2YWwiLCBzcGxpdCkpLmRvY19pZHMKICAgIG1hY3JvX2l0ZW1zID0gbnAuYXNhcnJheShwYW5lbFsibWFjcm9faXRlbXMiXSwgbnAuaW50NjQpCgogICAgZGVmIGl0ZW1zX29mKGtzKToKICAgICAgICBpZiBzb3J0ZWQoa3MpID09IHNvcnRlZChrZXlzKToKICAgICAgICAgICAgcmV0dXJuIG1hY3JvX2l0ZW1zICAgICAgICAgICAgICAgICAgICAgIyBleGFjdGFtZW50ZSBsb3Mgw610ZW1zIGRlbCBjb25maXJtYXRvcmlvCiAgICAgICAgcmV0dXJuIF9pbnRlcnNlY3Qoc3R1ZHksIGtzLCBzcGxpdCkKCiAgICByb3dzID0gY29tcHV0ZShncm91cHMsIGtleXMsIGl0ZW1zX29mLCBkb2NfaWRzLCBuYywgc3R1ZHkucm9wZV9wcCAvIDEwMCwKICAgICAgICAgICAgICAgICAgIGxhbWJkYSByZCwgaywgaXRlbXM6IF9hdChyZCwgZiJ0Z2VuX3trfSIsIGl0ZW1zKSwgX3RnZW1fYXQsIEI9QikKICAgIGF2diA9IGxvYWRfanNvbihzdHVkeS5wYXRocy5kKCJyZXN1bHRzIiwgcGhhc2UsICJhbGxfdnNfYWxsIiwgImFsbF92c19hbGwuanNvbiIpKSBvciB7fQogICAgYzFfY29uZiA9ICgoYXZ2LmdldCgiQzEiKSBvciB7fSkuZ2V0KCJwb3JfbWV0cmljYSIpIG9yIHt9KS5nZXQoInRnZW5tYWNybyIpCiAgICBjMl9jb25mID0gKGxvYWRfanNvbihzdHVkeS5wYXRocy5kKCJyZXN1bHRzIiwgcGhhc2UsICJDMiIsICJDMi5qc29uIikpIG9yIHt9KS5nZXQoInBhcmVzIikKICAgIHJlcyA9IHsibW90aXZvIjogTU9USVZPLCAiZmFzZSI6IHBoYXNlLCAic3BsaXQiOiBzcGxpdCwgInBhbmVsX3ZlcnNpb24iOiBwYW5lbC5nZXQoInZlcnNpb24iKSwgIm1hY3JvIjoga2V5cywKICAgICAgICAgICAiYnJhem9fQzEiOiBuYywgImNvcnJpZGFzIjoge2E6IGxlbih2KSBmb3IgYSwgdiBpbiBncm91cHMuaXRlbXMoKX0sICJyb3BlX3BwIjogc3R1ZHkucm9wZV9wcCwgIkIiOiBCLAogICAgICAgICAgICJmaWxhcyI6IHJvd3MsICJjb250cm9sIjogY29udHJvbChyb3dzLCBjMV9jb25mLCBjMl9jb25mKSwgImNyZWFkbyI6IG5vd19pc28oKSwgKioobWV0YSBvciB7fSl9CiAgICBkID0gbWFrZWRpcnMoc3R1ZHkucGF0aHMuZCgicmVzdWx0cyIsIHBoYXNlLCAic2Vuc2liaWxpZGFkX3BhbmVsIikpCiAgICBzYXZlX2pzb24ocmVzLCBvcy5wYXRoLmpvaW4oZCwgInNlbnNpYmlsaWRhZC5qc29uIikpCiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKGQsICJTRU5TSUJJTElEQUQubWQiKSwgInciLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGYud3JpdGUodG9fbWFya2Rvd24ocmVzKSkKICAgIGxvZyhmIltzZW5zaWJpbGlkYWRdIHtsZW4ocm93cyl9IHN1YmNvbmp1bnRvcyDCtyBjb250cm9sIHtyZXNbJ2NvbnRyb2wnXX0gwrcgZ3VhcmRhZG8gZW4ge2R9IikKICAgIHJldHVybiByZXMK"
SCRIPT_SHA256 = "c5a2a566c9b6c3318701a4b1fb0f6b113dc8e9a7fcf42d30b68f8e857e89f8fb"
src = base64.b64decode(SCRIPT_B64)
assert hashlib.sha256(src).hexdigest() == SCRIPT_SHA256, "el script embebido está corrupto"
os.makedirs("/content/analisis", exist_ok=True)
open("/content/analisis/sensibilidad_panel.py", "wb").write(src)
spec = importlib.util.spec_from_file_location("sensibilidad_panel", "/content/analisis/sensibilidad_panel.py")
sp = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sp)
study = setup(PROFILE, drive_base=DRIVE_BASE, cache_root=CACHE_ROOT)
if not os.path.exists(freeze_path(study)):
    raise SystemExit("En esta carpeta no hay PROTOCOL FREEZE: este análisis es sólo para F3. No se hizo nada.")
res = sp.panel_sensitivity(study, meta={"code_hash": CODE_HASH, "script_sha256": SCRIPT_SHA256, "notebook": "rev. A1 · 2026-10-07"})
if res:
    pd.set_option("display.width", 220); pd.set_option("display.max_columns", 20)
    display(sp.tabla(res))
    display(Markdown(open(os.path.join(study.paths.d("results", "F3", "sensibilidad_panel"), "SENSIBILIDAD.md"),
                          encoding="utf-8").read()))